<a href="https://cognitiveclass.ai"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0101EN-SkillsNetwork/images/IDSN-logo.png" width="400"> 

# Keras ile Evrişimli Sinir Ağları

Tahmini süre **30** dakika


Bu laboratuvarda, Keras kütüphanesini kullanarak evrişimli sinir ağları oluşturmayı öğreneceğiz. Ayrıca yaygın olarak kullanılan MNIST veri setini kullanacak ve elde ettiğimiz sonuçları geleneksel bir sinir ağıyla karşılaştıracağız.


## Bu Çalışma Kitabının Hedefleri    
* Keras kütüphanesini kullanarak evrişimli sinir ağları oluşturma
* Bir set evrişim ve havuzlama katmanına sahip evrişimli sinir ağı
* İki set evrişim ve havuzlama katmanına sahip evrişimli sinir ağı



## İçindekiler

<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 3>
      
1. <a href="#Import-Keras-and-Packages">Keras ve Paketleri İçe Aktarma   
2. <a href="#Convolutional-Neural-Network-with-One-Set-of-Convolutional-and-Pooling-Layers">Tek Set Evrişimli ve Havuzlama Katmanlarına Sahip Evrişimli Sinir Ağı  
3. <a href="#Convolutional-Neural-Network-with-Two-Sets-of-Convolutional-and-Pooling-Layers">İki Set Evrişimli ve Havuzlama Katmanına Sahip Evrişimli Sinir Ağı</a>  

</font>



### Gerekli kütüphaneleri yükleyin


Öncelikle, bir sinir ağı oluşturmak için ihtiyaç duyacağımız Keras kütüphanelerini ve paketleri yükleyelim.


In [1]:
# All Libraries required for this lab are listed below. The libraries need to be installed on Skills Network Labs. 
# If you run this notebook on a different environment, e.g. your desktop, you may want to install these.
!pip install numpy==2.0.2
!pip install pandas==2.2.2
!pip install tensorflow_cpu==2.18.0
!pip install matplotlib==3.9.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 173.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 MB 48.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 45.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 25.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#### TensorFlow uyarı mesajlarını gizleme
TensorFlow'da CPU mimarisinin kullanılması nedeniyle ortaya çıkan uyarı mesajlarını gizlemek için aşağıdaki kodu kullanıyoruz.
GPU mimarisini kullanıyorsanız, bu satırları **yorum satırına dönüştürmek** isteyebilirsiniz


In [25]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## Import Keras and Packages


In [26]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input
from keras.utils import to_categorical

Özellikle evrişimli sinir ağlarıyla çalışırken ek paketlere ihtiyacımız olacak.

In [27]:
from keras.layers import Conv2D # evrişimli katmanlar eklemek için
from keras.layers import MaxPooling2D # havuzlama katmanları eklemek için
from keras.layers import Flatten # tam bağlantılı katmanlar için verileri düzleştirmek için

## Bir Set Evrişimsel ve Toplama Katmanına Sahip Evrişimsel Sinir Ağı


In [18]:
# import data
from keras.datasets import mnist

# load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# reshape to be [samples][pixels][width][height]
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32')

Piksel değerlerini 0 ile 1 arasında olacak şekilde normalleştirelim

In [19]:
X_train = X_train / 255 # normalize training data
X_test = X_test / 255 # normalize test data

Şimdi, hedef değişkeni ikili kategorilere dönüştürelim


In [20]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

num_classes = y_test.shape[1] # kategori sayısı
print(num_classes)

10


Şimdi, modelimizi oluşturan bir fonksiyon tanımlayalım. Bir dizi evrişimli ve havuzlama katmanıyla başlayalım.

In [21]:
def convolutional_model():
    model = Sequential()  # Katmanların sırayla eklendiği bir CNN modeli oluşturur
    
    model.add(Input(shape=(28, 28, 1)))  
    # Modelin girdi boyutu: 28x28 piksel gri tonlamalı görüntü (MNIST gibi)
    
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation='relu'))  
    # 16 adet 5x5 filtre uygular → görüntüden özellik (feature) çıkarır
    # stride=(1,1) filtreyi her adımda 1 piksel kaydırır
    # ReLU aktivasyonu negatif değerleri sıfırlar ve non-linearity ekler
    
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))  
    # 2x2 max pooling uygular → feature map boyutunu küçültür
    # her 2x2 bölgedeki en büyük değeri alarak önemli bilgiyi korur
    
    model.add(Flatten())  
    # 2D feature map'i 1D vektöre çevirir
    # Fully connected katmanlara gönderebilmek için gereklidir
    
    model.add(Dense(100, activation='relu'))  
    # 100 nöronlu tam bağlı katman
    # CNN'den gelen özellikleri kullanarak daha karmaşık pattern öğrenir
    
    model.add(Dense(num_classes, activation='softmax'))  
    # Çıkış katmanı
    # num_classes kadar sınıf üretir (örn: 10 rakam)
    # softmax → olasılık dağılımı üretir
    
    # compile model
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])
    # optimizer='adam' → ağırlıkları güncelleyen optimizasyon algoritması
    # loss='categorical_crossentropy' → çok sınıflı sınıflandırma için kayıp fonksiyonu
    # metrics=['accuracy'] → model performansını doğruluk ile ölç
    
    return model  # Eğitilmeye hazır CNN modelini döndürür

Son olarak, modeli oluşturmak için işlevi çağıralım, ardından modeli eğitelim ve değerlendirelim.


In [22]:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
300/300 - 35s - 117ms/step - accuracy: 0.9176 - loss: 0.2912 - val_accuracy: 0.9653 - val_loss: 0.1152
Epoch 2/10
300/300 - 34s - 114ms/step - accuracy: 0.9722 - loss: 0.0947 - val_accuracy: 0.9782 - val_loss: 0.0650
Epoch 3/10
300/300 - 35s - 115ms/step - accuracy: 0.9811 - loss: 0.0649 - val_accuracy: 0.9818 - val_loss: 0.0553
Epoch 4/10
300/300 - 35s - 116ms/step - accuracy: 0.9851 - loss: 0.0490 - val_accuracy: 0.9831 - val_loss: 0.0514
Epoch 5/10
300/300 - 35s - 116ms/step - accuracy: 0.9875 - loss: 0.0411 - val_accuracy: 0.9861 - val_loss: 0.0408
Epoch 6/10
300/300 - 34s - 114ms/step - accuracy: 0.9899 - loss: 0.0329 - val_accuracy: 0.9869 - val_loss: 0.0404
Epoch 7/10
300/300 - 34s - 113ms/step - accuracy: 0.9917 - loss: 0.0273 - val_accuracy: 0.9862 - val_loss: 0.0418
Epoch 8/10
300/300 - 34s - 112ms/step - accuracy: 0.9933 - loss: 0.0225 - val_accuracy: 0.9881 - val_loss: 0.0364
Epoch 9/10
300/300 - 34s - 112ms/step - accuracy: 0.9947 - loss: 0.0181 - val_accuracy: 

------------------------------------------


## İki Grup Evrişimli ve Toplama Katmanına Sahip Evrişimli Sinir Ağı


Konvolüsyonel modelimizi, her birinden sadece birer katman yerine iki konvolüsyonel ve iki havuzlama katmanı içerecek şekilde yeniden tanımlayalım.


In [28]:
def convolutional_model():
    
    # create model
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    model.add(Conv2D(8, (2, 2), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    
    # Compile model
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])
    return model

Şimdi, yeni evrişimli sinir ağımızı oluşturmak için işlevi çağıralım; ardından onu eğitelim ve değerlendirelim.


In [29]:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
300/300 - 40s - 132ms/step - accuracy: 0.8678 - loss: 0.4676 - val_accuracy: 0.9568 - val_loss: 0.1455
Epoch 2/10
300/300 - 36s - 121ms/step - accuracy: 0.9621 - loss: 0.1273 - val_accuracy: 0.9728 - val_loss: 0.0864
Epoch 3/10
300/300 - 37s - 122ms/step - accuracy: 0.9727 - loss: 0.0907 - val_accuracy: 0.9737 - val_loss: 0.0800
Epoch 4/10
300/300 - 37s - 123ms/step - accuracy: 0.9781 - loss: 0.0716 - val_accuracy: 0.9818 - val_loss: 0.0621
Epoch 5/10
300/300 - 37s - 124ms/step - accuracy: 0.9818 - loss: 0.0596 - val_accuracy: 0.9831 - val_loss: 0.0564
Epoch 6/10
300/300 - 37s - 123ms/step - accuracy: 0.9843 - loss: 0.0514 - val_accuracy: 0.9832 - val_loss: 0.0504
Epoch 7/10
300/300 - 36s - 120ms/step - accuracy: 0.9861 - loss: 0.0454 - val_accuracy: 0.9858 - val_loss: 0.0451
Epoch 8/10
300/300 - 36s - 121ms/step - accuracy: 0.9877 - loss: 0.0401 - val_accuracy: 0.9881 - val_loss: 0.0385
Epoch 9/10
300/300 - 36s - 121ms/step - accuracy: 0.9887 - loss: 0.0371 - val_accuracy: 

<h3>Practice Exercise 1</h3>


Parti büyüklüğünün model eğitiminin süresine ve doğruluğuna nasıl etki ettiğini görelim.
Bunun için `batch_size` değerini 1024 olarak değiştirip, bunun doğruluk üzerindeki etkisini kontrol edebilirsiniz.


In [30]:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
59/59 - 36s - 617ms/step - accuracy: 0.6568 - loss: 1.2682 - val_accuracy: 0.8880 - val_loss: 0.3851
Epoch 2/10
59/59 - 35s - 586ms/step - accuracy: 0.9139 - loss: 0.2936 - val_accuracy: 0.9409 - val_loss: 0.2069
Epoch 3/10
59/59 - 35s - 588ms/step - accuracy: 0.9453 - loss: 0.1875 - val_accuracy: 0.9570 - val_loss: 0.1447
Epoch 4/10
59/59 - 34s - 580ms/step - accuracy: 0.9587 - loss: 0.1400 - val_accuracy: 0.9653 - val_loss: 0.1171
Epoch 5/10
59/59 - 34s - 568ms/step - accuracy: 0.9662 - loss: 0.1125 - val_accuracy: 0.9728 - val_loss: 0.0909
Epoch 6/10
59/59 - 34s - 568ms/step - accuracy: 0.9722 - loss: 0.0949 - val_accuracy: 0.9767 - val_loss: 0.0788
Epoch 7/10
59/59 - 34s - 579ms/step - accuracy: 0.9757 - loss: 0.0821 - val_accuracy: 0.9792 - val_loss: 0.0682
Epoch 8/10
59/59 - 34s - 572ms/step - accuracy: 0.9775 - loss: 0.0741 - val_accuracy: 0.9807 - val_loss: 0.0615
Epoch 9/10
59/59 - 34s - 574ms/step - accuracy: 0.9796 - loss: 0.0676 - val_accuracy: 0.9812 - val_loss:

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))
-->


<h3>Practice Exercise 2</h3>


Şimdi, epoch sayısının model eğitiminin süresini ve doğruluğunu nasıl etkilediğine bir bakalım.
Bunun için batch_size=1024 ve epochs=25 değerlerini koruyarak, bunun doğruluk üzerindeki etkisini kontrol edebilirsiniz.


In [31]:
model = convolutional_model()
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/25
59/59 - 36s - 604ms/step - accuracy: 0.6776 - loss: 1.2661 - val_accuracy: 0.8947 - val_loss: 0.3673
Epoch 2/25
59/59 - 35s - 596ms/step - accuracy: 0.9194 - loss: 0.2785 - val_accuracy: 0.9453 - val_loss: 0.1945
Epoch 3/25
59/59 - 34s - 575ms/step - accuracy: 0.9509 - loss: 0.1698 - val_accuracy: 0.9631 - val_loss: 0.1303
Epoch 4/25
59/59 - 34s - 576ms/step - accuracy: 0.9638 - loss: 0.1260 - val_accuracy: 0.9714 - val_loss: 0.1004
Epoch 5/25
59/59 - 34s - 576ms/step - accuracy: 0.9694 - loss: 0.1043 - val_accuracy: 0.9735 - val_loss: 0.0869
Epoch 6/25
59/59 - 34s - 583ms/step - accuracy: 0.9728 - loss: 0.0909 - val_accuracy: 0.9790 - val_loss: 0.0746
Epoch 7/25
59/59 - 35s - 597ms/step - accuracy: 0.9759 - loss: 0.0801 - val_accuracy: 0.9799 - val_loss: 0.0732
Epoch 8/25
59/59 - 34s - 582ms/step - accuracy: 0.9775 - loss: 0.0735 - val_accuracy: 0.9802 - val_loss: 0.0650
Epoch 9/25
59/59 - 33s - 566ms/step - accuracy: 0.9795 - loss: 0.0673 - val_accuracy: 0.9808 - val_loss:

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y;_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))


    -->


### Bu laboratuvarı tamamladığınız için teşekkür ederiz!

Bu not defteri [Alex Aklson](https://www.linkedin.com/in/aklson/) tarafından hazırlanmıştır. Umarım bu laboratuvarı ilginç ve öğretici bulmuşsunuzdur. Herhangi bir sorunuz olursa lütfen bana ulaşmaktan çekinmeyin!


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2024-11-20  | 3.0  | Aman  |  Updated the library versions to current |
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>

